In [ ]:
# Train the model on input data from data_input/ directory
# save data to a new output directory (which will hold the saved model parameters)
import os

# ---------------------------------------------------------
#  some general input parameters used later in the training
# ---------------------------------------------------------
#  
# pick your output directly
# default named for the etau scaling
odir = f'trained_model'

n_epochs = 500
xy_size = 60 # spatial size of the input data
last_time_step = 60 # last time step in the input data
nparams = 3 # trained number of parameters (energy density, vx, vy)
# note: if using flat input files, then the input size and organization must match 
# (nparams, xy_size, xy_size, last_time_step)

checkpoint_freq = 15 # these are quite expensive, so use an infrequently as possible
log_directly = True

events_per_file = 500
ver_event_ratio = 0.2 # ratio of events used for validation

batch_size = 10 # batch size for training

# can download some sample input files from xenodo at [file paths]
input_files = (
    './sample2K_0_10_flat_xy60_t60.root', # 0_10 = central
    './sample2K_40_60_flat_xy60_t60.root', # 40_60 = peripheral
    './sample2K_0_10_spikey_flat_xy60_t60.root', # central, spikey = nucleaon width = 0.8 fm
    './sample2K_40_60_spikey_flat_xy60_t60.root' # peripheral, spikey = nucleaon width = 0.8 fm
)

# short test files
n_epochs = 50
events_per_file = 50
input_files = (
'/home/davidstewart/JETSCAPE/config/bulk_writer/sample2K_0_10_spiky_flat_xy60_t60.root',
'/home/davidstewart/JETSCAPE/config/bulk_writer/sample2K_40_60_spiky_flat_xy60_t60.root',
'/home/davidstewart/JETSCAPE/config/bulk_writer/sample4K_0_10_flat_xy60_t60.root',
'/home/davidstewart/JETSCAPE/config/bulk_writer/sample100_0_10_nw9p6_flat_xy60_t60.root',
)


# download the files from xenodo at [file paths]
if not os.path.exists(odir):
    os.makedirs(odir)
log = open(f'{odir}/log.txt','w')

log.write('Input files:\n')
for input_file in input_files:
    log.write(f'  {input_file}\n')

log.write(f'number of epochs: {n_epochs}\n')
log.write(f'checkpoint frequency: {checkpoint_freq}\n')
log.write(f'last time step: {last_time_step}\n')
log.write(f'max events per input file: {events_per_file}\n')
log.write(f'nparams: {nparams}\n')
log.write(f'ratio of events to verify: {ver_event_ratio}\n')
log.write(f'batch size: {batch_size}\n')

In [ ]:
import uproot 
import awkward as ak
import numpy as np

####################
####################
####################

data = []
arr_train = []
arr_verify = []

for input_file in input_files:
    branch = uproot.open(input_file)['t']['user_res']
    nevents = min(events_per_file, branch.num_entries)
    arr = np.reshape(ak.to_numpy(branch.array(library='ak', entry_stop=nevents)), (nevents, nparams, xy_size, xy_size, last_time_step))

    n_verify = int(nevents * ver_event_ratio)
    n_train = nevents - n_train
    arr_train.append(arr[:n_train])
    arr_verify.append(arr[n_train:])

data_train = np.concatenate(arr_train, axis=0)
data_verify = np.concatenate(arr_verify, axis=0)

# randomly shuffle the events
np.random.shuffle(data_train)
np.random.shuffle(data_verify)


# scale the input array by tau:
tau0 = 0.5
taustep = 0.1

taus_scaling = tau0 + np.arange(data_train.shape[-1]) * taustep
data_train[:,0,:,:,:] *= taus_scaling

taus_scaling = tau0 + np.arange(data_verify.shape[-1]) * taustep
data_verify[:,0,:,:,:] *= taus_scaling

print(f'Training data shape: {data_train.shape}')
print(f'Verification data shape: {data_verify.shape}')

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, random_split, Dataset
import matplotlib.pyplot as plt
import sys
import awkward as ak
import os

sys.path.append('./loclibs')

In [ ]:
from neuralop.models import FNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.utils import count_model_params
from neuralop import LpLoss, H1Loss

# from neuralop.data.transforms.data_processors import DataProcessor, DefaultDataProcessor
# from neuralop.data.transforms.normalizers import UnitGaussianNormalizer, Normalizer
from neuralop.layers.embeddings import  GridEmbeddingND

In [ ]:
# Device configuration
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
#device = 'cpu'
#device = 'mps' # for Apple Sillcon somehow not converging on my macboock M1 as compared to running with CPU
print('Using device:', device)

In [ ]:
class FluidDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.time_steps = self.data.shape[-1] # take first time step as "real" input ....
        self.n_samples = self.data.shape[0]
        print( "Numpy data shape: ",self.data.shape )
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        x_initial = self.data[idx,:,:,:,:1]  
        x = np.repeat(x_initial, self.time_steps-1, axis=3) 
        y = self.data[idx,:,:,:,1:self.time_steps]  
        return {'x': torch.FloatTensor(x), 'y': torch.FloatTensor(y)}

    def values(self):
        return self.data.shape

In [ ]:
dataset_train = FluidDataset(data_train)
dataset_test = FluidDataset(data_verify)

train_loader = DataLoader(dataset_train, batch_size=10, shuffle=True)
test_loader = DataLoader(dataset_test, batch_size=10, shuffle=False)    

In [ ]:
dataset_train[0]['x'].shape, dataset_train[0]['x'].shape, dataset_test[0]['y'].shape, dataset_train[0]['y'].shape
dataset_test.values(), dataset_train.values()
test_resolutions=[100]  #change later to real resiolution of the data ...
test_batch_sizes=[10]

In [ ]:
test_loaders = {}
for res,test_bsize in zip(test_resolutions, test_batch_sizes):
        print("res",res)
        test_loaders[res] = DataLoader(dataset_test,
                                       batch_size=batch_size,
                                       shuffle=False,
                                       num_workers=0,
                                       pin_memory=True,
                                       persistent_workers=False,)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Model initialization
model = FNO(
    in_channels=nparams,        # Input channels (e.g., velocity field)
    out_channels=nparams,       # Output channels
    n_modes=[30,30,25],   # [60,60,50],    # Number of modes in each layer
    hidden_channels=64,   # 20             # Width of the network
    projection_channel_ratio=2
).to(device)

def calculate_model_memory(model):
    total_params = count_model_params(model) #sum(p.numel() for p in model.parameters())
    print(f'Total parameters: {total_params}')
    param_size = 4  # Size of a float32 in bytes
    total_memory = total_params * param_size  # Total memory in bytes
    
    # Convert to MB
    total_memory_MB = total_memory / (1024 ** 2)
    
    return total_memory_MB

memory_usage = calculate_model_memory(model)
print(f"Model memory usage: {memory_usage:.2f} MB")

#torch_info.summary(model, (1,100,11) , 100 , device) #not working with FNO model ...

In [ ]:
optimizer = AdamW(model.parameters(), lr=8e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

l2loss = LpLoss(d=3, p=2)
h1loss = H1Loss(d=3)

train_loss = h1loss
eval_losses={'h1': h1loss, 'l2': l2loss}

In [ ]:
trainer = Trainer(model=model, 
                  n_epochs=n_epochs,
                  device=device,
                  wandb_log=False,
                  eval_interval=2,
                  use_distributed=False,
                  verbose=True)
                  

In [ ]:
# custom code to periodically save the output model, event if not a local minimum
def custom_on_epoch_start(self, epoch):
    # Call the original method if needed
    # super(type(self), self).on_epoch_start(epoch)
    # Add your additional functionality here
    self.epoch = epoch

    if self.epoch % checkpoint_freq == 0 and self.epoch > 0:
        save_dir = f'{odir}/checkpoint_{self.epoch}'
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
        self.checkpoint(save_dir)
        print('saving checkpoint to ', save_dir)

trainer.on_epoch_start = custom_on_epoch_start.__get__(trainer, type(Trainer))

In [ ]:
import sys
import atexit

# Redirect stdout to a log file to capture step-by-step output
log_path = f'{odir}/training.log'

if log_directly:
    class Logger(object):
        def __init__(self, filename):
            self.terminal = sys.stdout
            self.log = open(filename, "w")
        def write(self, message):
            self.terminal.write(message)
            self.log.write(message)
        def flush(self):
            self.terminal.flush()
            self.log.flush()

    sys.stdout = Logger(log_path)

    def restore_stdout():
        sys.stdout.log.close()
        sys.stdout = sys.stdout.terminal
    atexit.register(restore_stdout)

trainer.train(train_loader=train_loader,
              test_loaders=test_loaders,
              optimizer=optimizer,
              scheduler=scheduler, 
              regularizer=False, 
              save_dir=odir,
              save_best='100_l2',
              training_loss=train_loss,
              eval_losses=eval_losses)

# END OF TRAINING

The remaining cells do some diagnostics:
 1. They plot the progression of the training_loss and eval_losses values through each epoch
 2. They can load the model, and then take some of the verification data, apply the model 
    on it, and generate some PDFs of how good the model is
 3. They can calculate the average radial and transverse velocity profiles of the prediced data to the truth data
    (again, using verification events)

In [ ]:
# plot the training parameters
import sys
sys.path.append('./loc_libs')
from parse_training_log import plot_training_log

if not last_time_step:
    last_time_step = 60# data_train.shape[-1]  # Use the last time step from the training data
data = plot_training_log(f'{odir}/training.log',f'{n_epochs} epochs, {nevents} events, {last_time_step} time steps', first_entry=0, times=10)


In [ ]:
import torch

model = FNO(
    in_channels=nparams,        # Input channels (e.g., velocity field)
    out_channels=nparams,       # Output channels
    # positional_embedding=GridEmbeddingND(in_channels=3, dim=3, grid_boundaries=[[-15,15],[-15,15],[3.5,3.5001]]),
    #  positional_embedding=GridEmbeddingND(in_channels=3, dim=3, grid_boundaries=[[-15,15],[-15,15],[0.6,15.5]]),
    n_modes=[30,30,25],   # [60,60,50],    # Number of modes in each layer
    hidden_channels=64,   # 20             # Width of the network
    projection_channel_ratio=2
).to(device)

model.load_state_dict(torch.load(os.path.join(odir, 'best_model_state_dict.pt')))

In [ ]:
''' correct the testing data, and save the data to the outputs '''
# test the model input
def get_xy(in_data, i):
    # get the input and output data for the model

    time_steps=in_data.shape[-1]

    x = in_data[i,:,:,:,:1]
    x = np.repeat(x, time_steps-1,axis=3)
    y = in_data[i,:,:,:,1:time_steps]

    x = torch.FloatTensor(x)
    y = torch.FloatTensor(y)

    return x, y

def model_result(model, data, model_print=True):
    x_out = []
    y_out = []
    model_out = []
    print(data.shape)
    print(data[0].shape)
    for i in range(data.shape[0]):
        x, y = get_xy(data, i)
        if i == 0:
            print('x shape:', x.shape)
            print('y shape:', y.shape)
        xin = x.unsqueeze(0).to(device)
        out = model(xin).detach().cpu().numpy()
        x_out.append( x[:,:,:,0].cpu().numpy() )
        y_out.append(y[:,:,:,:].cpu().numpy())
        # y_out.append(_y_out[..., np.newaxis])
        model_out.append( out[0] )

    x_out = np.stack(x_out)
    y_out = np.stack(y_out)
    model_out = np.stack(model_out)

    if model_print:
        print('x_out shape:', x_out.shape)
        print('y_out shape:', y_out.shape)
        print('model_out shape:', model_out.shape)

    return {'x':x_out, 'y':y_out, 'model':model_out}

pred = model_result(model, data_verify, model_print=True)
print('x',pred['x'].shape)
print('y',pred['y'].shape)
print('model',pred['model'].shape)

# inverse the etau transformation
dat_modeled = pred['model']
tau0 = 0.5
tau_step = 0.1
tau_scaling = tau0 + np.arange(dat_modeled.shape[-1]) * tau_step
dat_modeled[:,0,:,:,:] /= tau_scaling

dat_truth = pred['y']
dat_truth[:,0,:,:,:] /= tau_scaling

In [ ]:
# test some output:
# truth = np.load(input_raw)['verify'][:,:,:,:,1:]
print('truth shape:', dat_truth.shape)
print('model shape:', dat_modeled.shape)

sys.path.append('loclibs')
from contour_v_plot import plot_three_bins_contour

for i_event in range(5):
    plot_three_bins_contour(dat_modeled, dat_truth, iT=(0, 30, 58), event=i_event, save=f'{odir}/contour_{i_event}.pdf') # can also add if wanted to save the PDFs: save=f'{odir}/contour_{i_event}.pdf'))